# PUBG API Data Extraction & Structuring (Python Version)
This notebook tests the extraction of PUBG match data and structures it into flattened Pandas DataFrames. It includes high-resolution telemetry data.

In [2]:
import requests
import pandas as pd
import json
import time
from config import api_key
from IPython.display import display

In [3]:
valid_shards = ["steam", "console"]

### Steam Data Extraction

In [6]:
##Fetches a list of recent match IDs from the PUBG API (It is the only way to get a random, high-volume list of matches without knowing specific players beforehand)

def get_pubg_samples_steam(api_key, shard=valid_shards[0]):
    url = f"https://api.pubg.com/shards/{shard}/samples"
    headers = {
        "Authorization": f"Bearer {api_key}", 
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [7]:
match_samples = get_pubg_samples_steam(api_key)

print(match_samples)


{'data': {'type': 'sample', 'id': '43272d88-7192-448b-9551-1b1ce92068e0', 'attributes': {'createdAt': '2026-04-30T00:00:00Z', 'titleId': 'bluehole-pubg', 'shardId': 'steam'}, 'relationships': {'matches': {'data': [{'type': 'match', 'id': '95ff792e-3728-49c8-9ca2-6e1364af7639'}, {'type': 'match', 'id': '647279b3-e708-4039-a378-03661d589c8e'}, {'type': 'match', 'id': '35a539fb-b3c4-4d4c-95cd-acc20c4c1d23'}, {'type': 'match', 'id': '91cbeac6-fc0d-4a5c-8f14-cbe37bef108d'}, {'type': 'match', 'id': '6333c7e7-8ea2-4807-aee2-8af661c512a1'}, {'type': 'match', 'id': '6dfa8e22-edf2-4e69-a40e-38f6b442bd8d'}, {'type': 'match', 'id': '246e4b32-b5c3-4859-9212-5cfef00a1e26'}, {'type': 'match', 'id': '20b8328f-d085-4cde-8087-cf1ceb8755a0'}, {'type': 'match', 'id': '393bfc6a-b525-4672-ad5e-74a16b34a293'}, {'type': 'match', 'id': '6be7a9b1-e3a9-4803-a05f-582107b02265'}, {'type': 'match', 'id': '74d8703a-8b34-4317-9df0-9cb20fb53215'}, {'type': 'match', 'id': '326e1c78-4d80-40c8-b25e-70a0c8b2c2ed'}, {'type

In [11]:
matches_list = match_samples['data']['relationships']['matches']['data']
match_samples_df = pd.DataFrame(matches_list)
print(f"Total matches retrieved: {len(match_samples_df)}")
display(match_samples_df.head())

Total matches retrieved: 1016


,type,id
0,match,95ff792e-3728-49c8-9ca2-6e1364af7639
1,match,647279b3-e708-4039-a378-03661d589c8e
2,match,35a539fb-b3c4-4d4c-95cd-acc20c4c1d23
3,match,91cbeac6-fc0d-4a5c-8f14-cbe37bef108d
4,match,6333c7e7-8ea2-4807-aee2-8af661c512a1


In [17]:
match_id = match_samples_df['match_id'].tolist()

In [19]:
def get_match_details(api_key, match_id, shard=valid_shards[0]):
    """Fetch detailed information for a specific match."""
    url = f"https://api.pubg.com/shards/{shard}/matches/{match_id}"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [25]:
match_details = get_match_details(api_key,match_id[0])
print(match_details)

{'data': {'type': 'match', 'id': '95ff792e-3728-49c8-9ca2-6e1364af7639', 'attributes': {'shardId': 'steam', 'tags': None, 'isCustomMatch': False, 'createdAt': '2026-04-29T23:42:44Z', 'stats': None, 'titleId': 'bluehole-pubg', 'mapName': 'Tiger_Main', 'matchType': 'official', 'seasonState': 'progress', 'duration': 1532, 'gameMode': 'duo-fpp'}, 'relationships': {'assets': {'data': [{'type': 'asset', 'id': 'f551103a-4428-11f1-ba3a-667ef378d19f'}]}, 'rosters': {'data': [{'type': 'roster', 'id': 'f7cb0d8f-280b-49a2-9339-8a8c1f915e2f'}, {'type': 'roster', 'id': '3e38ea4b-d93e-4cc4-9366-dd38f7f0f7c5'}, {'type': 'roster', 'id': '1d774de1-26cf-4b12-9238-3829a4e0517f'}, {'type': 'roster', 'id': 'c870f0a6-073e-4cef-b64c-9ef917c99e46'}, {'type': 'roster', 'id': '983fcbf5-b6c8-4db7-8b6a-52f5a1e4e06b'}, {'type': 'roster', 'id': '208ce643-c1b5-4694-9de1-8f429d09c131'}, {'type': 'roster', 'id': 'c358ccd3-4ca6-46d1-bfef-4e216c6df11f'}, {'type': 'roster', 'id': '68f77f4f-03f2-4411-b96b-a1fea896c902'}, {

In [21]:
match_details = get_match_details(api_key,match_id[0])
matches_details_list = match_details['data']['relationships']['assets']['data']
print(matches_details_list)

[{'type': 'asset', 'id': 'f551103a-4428-11f1-ba3a-667ef378d19f'}]


In [39]:
def extract_telemetry_url(match_details):
    included_items = match_details.get("included", [])
    
    for item in included_items:
        is_telemetry = item.get("attributes", {}).get("name") == "telemetry"
        
        if is_telemetry:
            telemetry_url = item["attributes"]["URL"]
            print("Telemtry URL sucessfully extracted!")
            return telemetry_url
            
    print("Did not find telemetry URL in match details:(")
    return None

In [ ]:
telemetry_url = extract_telemetry_url(match_details)

Telemtry URL sucessfully extracted!


In [41]:
telemetry_url_df = pd.DataFrame(telemetry_url)
display(telemetry_url_df.head())

ValueError: DataFrame constructor not properly called!